# 10 - Spark from Scala (Spark Connect)

Run distributed Spark from **Scala** against the in-stack `spark-connect`
sidecar — the Scala counterpart to `09_spark_connect.ipynb` (Python).

## Prerequisites

- Open this notebook with the **Scala 2.13** kernel (id `scala213`). Spark 4.x
  is built for Scala 2.13 only, so the Connect client has no Scala 3 artifact.
- `SPARK_SOURCE != disabled` (the sidecar must be up). The first `import $ivy`
  below downloads the Connect client + gRPC/protobuf deps into
  `~/.cache/coursier/` — give it a minute on first run.

## 1. Load the Spark Connect Scala client
`import $ivy` (coursier) pulls the Maven artifact at runtime — no image change.

In [ ]:
import $ivy.`org.apache.spark::spark-connect-client-jvm:4.1.2`

## 2. Connect
Reads `SPARK_REMOTE` (compose-injected, default `sc://spark-connect:15002`) —
the same knob the Python notebook uses, so both target the same server.

In [ ]:
import org.apache.spark.sql.SparkSession

val sparkRemote = sys.env.getOrElse("SPARK_REMOTE", "sc://spark-connect:15002")
val spark = SparkSession.builder().remote(sparkRemote).getOrCreate()
println(s"connected: $sparkRemote | Spark ${spark.version}")

## 3. DataFrame + SQL
These execute on the remote driver, not in this kernel.

In [ ]:
import spark.implicits._
import org.apache.spark.sql.functions.col

val df = spark.range(10).withColumn("squared", col("id") * col("id"))
df.show()
println(s"count: ${df.count()}")
spark.sql("SELECT 1 + 1 AS result").show()

## 4. Done
`s3a://` reads/writes work the same as the Python path (storage creds live on
the server). Stop the session when finished.

In [ ]:
spark.stop()